# T12: Validation Gates

## What You'll Learn

Validation gates are automated checks that run against your data to catch quality issues before they reach downstream consumers. In this notebook you will:

1. **Understand** the gate framework and its components
2. **Run** a referential integrity gate
3. **Run** a null constraint gate
4. **Run** a unique constraint gate
5. **Execute** the full gate runner with all gates at once
6. **Demonstrate** the quarantine flow for failed records

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle`

## Time Estimate

**~5 minutes** from start to finish.

In [ ]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## The Gate Framework

### What are validation gates?

A validation gate is a single, focused check that answers one question about your data:

| Gate | Question It Answers |
|------|--------------------|
| **ReferentialIntegrityGate** | Do all foreign keys point to valid parent records? |
| **NullConstraintGate** | Are NOT NULL columns free of nulls? |
| **UniqueConstraintGate** | Are unique/primary key columns actually unique? |

### How gates work

1. You create a `ValidationContext` containing your tables and schema
2. Each gate's `check()` method inspects the context and returns a result
3. The result says `passed=True` or `passed=False` with details about violations
4. The `GateRunner` orchestrates multiple gates and produces a summary

### Why this matters

Gates are the safety net between data generation (or ingestion) and consumption. They catch problems early — before bad data poisons your dashboards, ML models, or reports.

## Generate Clean Data

### What we're about to do
We'll generate a clean retail dataset and create a `ValidationContext` from it. Clean data should pass all gates. We'll verify that first, then intentionally break things to see gates catch violations.

### Why this matters
Testing gates against clean data confirms they don't produce false positives. A gate that flags clean data is worse than no gate at all.

### What to expect
A clean dataset and a validation context ready for gate checks.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain
from sqllocks_spindle.validation import (
    GateRunner,
    ValidationContext,
    ReferentialIntegrityGate,
    NullConstraintGate,
    UniqueConstraintGate,
)

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())

# Create a validation context
ctx = ValidationContext(tables=result.tables, schema=result.schema)
print(f"\nValidation context created with {len(ctx.tables)} tables.")

## Run the Referential Integrity Gate

### What we're about to do
We'll run the `ReferentialIntegrityGate` against our clean data. This gate checks every foreign-key relationship defined in the schema and verifies that all child records reference valid parent records.

### Why this matters
Orphaned records (child rows with no matching parent) cause silent data loss in JOINs. This gate catches them before they reach your queries.

### What to expect
A PASS result, since Spindle generates relationally consistent data.

In [ ]:
ri_gate = ReferentialIntegrityGate()
ri_result = ri_gate.check(ctx)

print(f"=== Referential Integrity Gate ===")
print(f"Status: {'PASS' if ri_result.passed else 'FAIL'}")
print(f"Checks performed: {ri_result.checks_performed}")
print(f"Violations: {ri_result.violation_count}")

if ri_result.details:
    print(f"\nDetails:")
    for detail in ri_result.details:
        print(f"  {detail}")

## Run the Null Constraint Gate

### What we're about to do
We'll run the `NullConstraintGate` which checks that columns marked as NOT NULL in the schema contain no null values.

### Why this matters
Unexpected nulls are the most common data quality issue. They cause calculation errors, broken aggregations, and misleading visualizations. This gate catches them at the source.

### What to expect
A PASS result for clean data.

In [ ]:
null_gate = NullConstraintGate()
null_result = null_gate.check(ctx)

print(f"=== Null Constraint Gate ===")
print(f"Status: {'PASS' if null_result.passed else 'FAIL'}")
print(f"Checks performed: {null_result.checks_performed}")
print(f"Violations: {null_result.violation_count}")

if null_result.details:
    print(f"\nDetails:")
    for detail in null_result.details:
        print(f"  {detail}")

## Run the Unique Constraint Gate

### What we're about to do
We'll run the `UniqueConstraintGate` which verifies that primary key and unique columns contain no duplicate values.

### Why this matters
Duplicate keys cause incorrect JOIN fan-outs, double-counting in aggregations, and subtle bugs that are hard to trace. This gate ensures every key is truly unique.

### What to expect
A PASS result for clean data.

In [ ]:
unique_gate = UniqueConstraintGate()
unique_result = unique_gate.check(ctx)

print(f"=== Unique Constraint Gate ===")
print(f"Status: {'PASS' if unique_result.passed else 'FAIL'}")
print(f"Checks performed: {unique_result.checks_performed}")
print(f"Violations: {unique_result.violation_count}")

if unique_result.details:
    print(f"\nDetails:")
    for detail in unique_result.details:
        print(f"  {detail}")

## Run the Full Gate Runner

### What we're about to do
Instead of running gates individually, we'll use the `GateRunner` to execute all gates in sequence and produce a unified report. This is the pattern you'd use in a production pipeline.

### Why this matters
The gate runner gives you a single pass/fail decision for your entire dataset. It also collects all violations across all gates, making it easy to generate a quality report or trigger alerts.

### What to expect
An overall PASS result with a summary of all gates checked.

In [ ]:
runner = GateRunner(
    gates=[ReferentialIntegrityGate(), NullConstraintGate(), UniqueConstraintGate()]
)

runner_result = runner.run(ctx)

print(f"=== Gate Runner Summary ===")
print(f"Overall status: {'PASS' if runner_result.all_passed else 'FAIL'}")
print(f"Gates run: {runner_result.gates_run}")
print(f"Gates passed: {runner_result.gates_passed}")
print(f"Gates failed: {runner_result.gates_failed}")

print(f"\n=== Per-Gate Results ===")
for gate_name, gate_result in runner_result.results.items():
    status = "PASS" if gate_result.passed else "FAIL"
    print(f"  {gate_name}: {status} ({gate_result.violation_count} violations)")

## Quarantine Flow: Inject Chaos and Catch It

### What we're about to do
We'll intentionally corrupt the data using Spindle's chaos engine, then run the gate runner again. This time, gates should FAIL. We'll then show how to quarantine the bad records.

### Why this matters
This is the full lifecycle: generate data, corrupt it (simulating production issues), run validation gates, and quarantine failures. It proves your quality checks work end-to-end.

### What to expect
One or more gates will FAIL, and we'll extract the specific records that caused the failure.

In [ ]:
from sqllocks_spindle.chaos import ChaosConfig, ChaosEngine
import pandas as pd

# Apply chaos to create bad data
chaos_config = ChaosConfig(enabled=True, intensity="stormy", seed=99)
engine = ChaosEngine(chaos_config)

# Corrupt copies of the tables
corrupted_tables = {}
for name, df in result.tables.items():
    corrupted_tables[name] = engine.corrupt_dataframe(df.copy(), day=5)

# Create a new validation context with corrupted data
corrupted_ctx = ValidationContext(tables=corrupted_tables, schema=result.schema)

# Run all gates
corrupted_result = runner.run(corrupted_ctx)

print(f"=== Gate Runner on Corrupted Data ===")
print(f"Overall status: {'PASS' if corrupted_result.all_passed else 'FAIL'}")
print(f"Gates passed: {corrupted_result.gates_passed}")
print(f"Gates failed: {corrupted_result.gates_failed}")

print(f"\n=== Per-Gate Results ===")
for gate_name, gate_result in corrupted_result.results.items():
    status = "PASS" if gate_result.passed else "FAIL"
    print(f"  {gate_name}: {status} ({gate_result.violation_count} violations)")
    if not gate_result.passed and gate_result.details:
        for detail in gate_result.details[:3]:
            print(f"    -> {detail}")

## Quarantine Bad Records

### What we're about to do
We'll separate the bad records from the good ones — quarantining rows that violate null constraints so they can be reviewed and fixed without blocking the rest of the pipeline.

### Why this matters
Quarantining is better than rejecting an entire batch. Good records flow through to downstream consumers, while bad records go to a quarantine table for investigation and remediation.

### What to expect
A clean DataFrame and a quarantine DataFrame, with row counts showing the split.

In [ ]:
# Demonstrate quarantine on the customers table
# Find rows with nulls in columns that should be NOT NULL
customers_corrupted = corrupted_tables["customers"]
customers_original = result.tables["customers"]

# Identify NOT NULL columns from the original data (columns that had zero nulls)
not_null_cols = [col for col in customers_original.columns
                 if customers_original[col].isnull().sum() == 0]

# Split into clean and quarantine
has_violation = customers_corrupted[not_null_cols].isnull().any(axis=1)
clean_df = customers_corrupted[~has_violation].copy()
quarantine_df = customers_corrupted[has_violation].copy()

print(f"=== Quarantine Results ===")
print(f"Total records: {len(customers_corrupted)}")
print(f"Clean records: {len(clean_df)} ({len(clean_df)/len(customers_corrupted)*100:.1f}%)")
print(f"Quarantined records: {len(quarantine_df)} ({len(quarantine_df)/len(customers_corrupted)*100:.1f}%)")

if len(quarantine_df) > 0:
    print(f"\n=== Sample Quarantined Records ===")
    print(quarantine_df.head(3).to_string())

    print(f"\n=== Null Columns in Quarantined Records ===")
    for col in not_null_cols:
        null_count = quarantine_df[col].isnull().sum()
        if null_count > 0:
            print(f"  {col}: {null_count} nulls")

## What's Next?

You've built a complete validation pipeline — from clean data through chaos injection to gate checking and quarantine. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T11: Chaos Engineering** | Dive deeper into chaos injection at different intensity levels |
| **T10: File-Drop Simulation** | Simulate daily file drops with manifests and late arrivals |
| **T07: Fabric Lakehouse Quick Start** | Write Parquet and Delta Lake files for Fabric Lakehouse |

Happy validating!